In [ ]:
%load_ext autoreload
%autoreload 2

# Funnel du choix de la convention collective sur les contributions

Analyse du bloc « Quelle est votre convention collective ? » affiché sur les contributions
(fiche générique et fiche CC), à partir des 24 events de la catégorie Matomo `cc_search_funnel`
posés par la PR [#7469](https://github.com/SocialGouv/code-du-travail-numerique/pull/7469)
(données disponibles à partir du **2026-09-08**).

Objectif : voir, entre l'affichage du bloc et le clic sur « Afficher les informations », **où l'on
perd les usagers** et pour quelles raisons (recherche sans résultat, clic bloqué, entreprise sans CC,
retour arrière…).

**Attention : cette analyse s'appuie sur les données de visites (via la BDD extraite). Il faut donc
avoir accès à ces informations pour l'utiliser (non compatible RGPD).**

## Rappel des events (catégorie `cc_search_funnel`)

Le `name` de chaque event porte le chemin de la page : `contribution/<slug>` sur une fiche générique,
`contribution/<num>-<slug>` (ou `contribution/<slug>/<num>-<slug>`) sur une fiche CC.

| Étape | Action | Quand |
| --- | --- | --- |
| Dénominateur | `view_bloc_cc` | Montage du bloc de sélection |
| Aparté | `click_c_est_quoi_une_cc` | Clic sur « La convention collective, c'est quoi ? » |
| Choix du parcours | `select_p1` / `select_p2` / `select_p3` | Clic sur la radio (je connais ma CC / je cherche mon entreprise / je ne souhaite pas renseigner) |
| p1 | `start_recherche_cc`, `no_result_cc` | 1re frappe dans l'autocomplete CC ; recherche sans résultat |
| p2 | `start_recherche_entreprise`, `submit_recherche_entreprise`, `select_localisation`, `no_result_entreprise`, `error_recherche_entreprise` | Saisie, soumission(s), localisation, échec, incident API |
| p2 | `select_entreprise`, `select_particulier_employeur`, `entreprise_sans_cc`, `select_cc_entreprise` | Entreprise choisie, raccourci particulier employeur, impasse, choix d'une CC parmi N |
| Retours arrière | `click_modifier_entreprise`, `click_modifier_cc` | Boutons « Modifier » |
| Fin | `click_afficher_les_informations` | **Toute** tentative de clic sur le bouton principal |
| Bloqué | `blocked_sans_option`, `blocked_sans_cc_p1`, `blocked_sans_cc_p2` | Clic sans option / sans CC en p1 / sans CC en p2 |
| Alerte | `cc_non_traitee_retenue`, `click_lien_cc_externe` | CC sans réponse sur cette contribution ; clic vers Légifrance |

## Hypothèses de lecture

- **Unité d'analyse = un parcours = (visite, page)**, c'est-à-dire un `idvisit` sur un `name`
  donné. Une visite qui consulte deux contributions compte deux parcours. Seuls les parcours ayant
  au moins un `view_bloc_cc` sont conservés.
- Chaque étape est comptée **une fois par parcours** (booléen « l'a fait au moins une fois »).
- Le parcours (p1/p2/p3) d'un clic « Afficher les informations » est déduit de la **dernière radio
  cochée avant le clic** dans la séquence des events. Sans radio cochée avant (cas des fiches CC où
  le parcours est pré-coché, aucun `select_pX` n'est alors émis), le clic est rangé en
  `route non identifiée`.
- Un clic est **réussi** si le parcours compte plus de `click_afficher_les_informations` que de
  `blocked_*` sur la même route (les deux events partent dans le même handler et l'ordre Matomo
  à la même seconde n'est pas garanti, on ne peut donc pas les apparier un à un).
- `view_bloc_cc` n'a pas la même nature sur les deux façades (affiché à chaque page vue sur la
  fiche générique, uniquement à l'ouverture explicite du bloc sur une fiche CC) : le funnel est
  aussi décliné par façade plus bas.

## 1. Extraction des events

In [ ]:
import re
import pandas as pd
from analysis.connectors.matomo import MatomoSQLConnectorSync

interval_start = '2026-09-08 00:00:00'   # mise en prod des events cc_search_funnel
interval_stop  = '2026-09-15 00:00:00'   # borne haute exclue

columns = ['idvisit', 'action_id', 'action_timestamp', 'operatingsystemname',
           'action_eventaction', 'action_eventname']

query = f"""
    SELECT {", ".join(columns)}
    FROM matomo_partitioned
    WHERE action_timestamp >= '{interval_start}'
      AND action_timestamp <  '{interval_stop}'
      AND action_type = 'event'
      AND action_eventcategory = 'cc_search_funnel'
"""

with MatomoSQLConnectorSync() as matomo:
    events_df = matomo.run_query_df(query)

print(f"{len(events_df)} events, {events_df['idvisit'].nunique()} visites")
events_df['action_eventaction'].value_counts()

## 2. Préparation

- `seq` : rang de l'action dans la visite (suffixe de `action_id`, plus fiable que le timestamp à la
  seconde près) ;
- `page` : le `name` de l'event, qui identifie la contribution ;
- `facade` : fiche générique ou fiche CC (présence d'un `<num>-` dans le chemin) ;
- `device` : mobile / desktop d'après l'OS.

In [ ]:
events_df['seq'] = events_df['action_id'].str.rsplit('_', n=1).str[1].astype(int)
events_df = events_df.rename(columns={'action_eventaction': 'action', 'action_eventname': 'page'})
events_df = events_df.sort_values(['idvisit', 'seq']).reset_index(drop=True)

def to_facade(page):
    return 'fiche CC' if re.search(r'/\d+-', page or '') else 'générique'

mobile_os  = {'Android', 'iOS'}
desktop_os = {'Windows', 'Mac', 'GNU/Linux', 'Chrome OS', 'Ubuntu'}

def to_device(os):
    if os in mobile_os:
        return 'mobile'
    if os in desktop_os:
        return 'desktop'
    return 'autre'

events_df['facade'] = events_df['page'].apply(to_facade)
events_df['device'] = events_df['operatingsystemname'].apply(to_device)

print(events_df['facade'].value_counts(), end='\n\n')
print(events_df['device'].value_counts())

## 3. Construction des parcours

On parcourt chaque parcours (visite × page) dans l'ordre des events. On compte chaque action, et on
attribue les events dépendants du parcours (`click_afficher_les_informations`, `click_modifier_cc`,
`cc_non_traitee_retenue`) à la **dernière radio cochée** (`p1`, `p2`, `p3`, ou `none`).

In [ ]:
from collections import Counter

ROUTE_OF = {'select_p1': 'p1', 'select_p2': 'p2', 'select_p3': 'p3'}
ROUTED   = {'click_afficher_les_informations': 'click',
            'click_modifier_cc':               'modifier_cc',
            'cc_non_traitee_retenue':          'untreated'}

def scan(actions, route='none'):
    """Compte les actions ; `route` = parcours courant au début de la séquence."""
    c = Counter()
    for a in actions:
        c[a] += 1
        if a in ROUTE_OF:
            route = ROUTE_OF[a]
        elif a in ROUTED:
            c[f'{ROUTED[a]}_{route}'] += 1
    return c

records = []
for (idvisit, page), grp in events_df.groupby(['idvisit', 'page'], sort=False):
    rec = {'idvisit': idvisit, 'page': page,
           'facade': grp['facade'].iloc[0], 'device': grp['device'].iloc[0]}
    rec.update(scan(grp['action'].tolist()))
    records.append(rec)

counts_df = pd.DataFrame(records).fillna(0)
meta_cols = ['idvisit', 'page', 'facade', 'device']
for col in counts_df.columns.difference(meta_cols):
    counts_df[col] = counts_df[col].astype(int)

# Garantit la présence de toutes les colonnes, même si un event n'est jamais apparu sur la période
ALL_COUNTERS = list(ROUTE_OF) + [
    'view_bloc_cc', 'click_c_est_quoi_une_cc', 'start_recherche_cc', 'no_result_cc',
    'start_recherche_entreprise', 'submit_recherche_entreprise', 'select_localisation',
    'no_result_entreprise', 'error_recherche_entreprise', 'select_entreprise',
    'entreprise_sans_cc', 'select_cc_entreprise', 'select_particulier_employeur',
    'click_modifier_entreprise', 'click_modifier_cc', 'click_afficher_les_informations',
    'blocked_sans_option', 'blocked_sans_cc_p1', 'blocked_sans_cc_p2',
    'cc_non_traitee_retenue', 'click_lien_cc_externe',
] + [f'{p}_{r}' for p in ROUTED.values() for r in ('p1', 'p2', 'p3', 'none')]
for col in ALL_COUNTERS:
    if col not in counts_df:
        counts_df[col] = 0

# On ne garde que les parcours dont on a vu le bloc (dénominateur du funnel)
without_view = counts_df[counts_df['view_bloc_cc'] == 0]
counts_df = counts_df[counts_df['view_bloc_cc'] > 0].reset_index(drop=True)

print(f"Parcours (visite × page)        : {len(counts_df) + len(without_view)}")
print(f"Parcours conservés (bloc vu)    : {len(counts_df)}")
print(f"Parcours écartés (bloc non vu)  : {len(without_view)}")

## 4. Étapes du funnel

Une ligne = un parcours, une colonne = une étape (booléen « atteinte au moins une fois »).

In [ ]:
def to_steps(c):
    """Booléens d'étapes à partir des compteurs d'un ensemble de parcours."""
    return pd.DataFrame({
        # Dénominateur
        'Start':      c['view_bloc_cc'] > 0,
        'Any_route':  (c['select_p1'] + c['select_p2'] + c['select_p3']) > 0,
        # Parcours p1 : je connais ma CC
        'P1':         c['select_p1'] > 0,
        'P1_search':  (c['select_p1'] > 0) & (c['start_recherche_cc'] > 0),
        'P1_click':   c['click_p1'] > 0,
        'P1_ok':      c['click_p1'] > c['blocked_sans_cc_p1'],
        # Parcours p2 : je cherche mon entreprise
        'P2':         c['select_p2'] > 0,
        'P2_search':  (c['select_p2'] > 0) & (c['start_recherche_entreprise'] > 0),
        'P2_submit':  (c['select_p2'] > 0) & (c['submit_recherche_entreprise'] > 0),
        'P2_select':  (c['select_entreprise'] + c['select_particulier_employeur']) > 0,
        'P2_click':   c['click_p2'] > 0,
        'P2_ok':      c['click_p2'] > c['blocked_sans_cc_p2'],
        # Parcours p3 : je ne souhaite pas renseigner ma CC (jamais bloqué)
        'P3':         c['select_p3'] > 0,
        'P3_click':   c['click_p3'] > 0,
        'P3_ok':      c['click_p3'] > 0,
        # Clics sans radio cochée dans la séquence (parcours pré-coché sur fiche CC, ou clic à vide)
        'None_click': c['click_none'] > 0,
        'None_ok':    c['click_none'] > c['blocked_sans_option'],
        # Succès global, toutes routes confondues
        'Click_any':  c['click_afficher_les_informations'] > 0,
        'OK_any':     c['click_afficher_les_informations']
                      > (c['blocked_sans_option'] + c['blocked_sans_cc_p1'] + c['blocked_sans_cc_p2']),
        # Décrochages / apartés
        'what_is_cc':           c['click_c_est_quoi_une_cc'] > 0,
        'blocked_sans_option':  c['blocked_sans_option'] > 0,
        'no_result_cc':         c['no_result_cc'] > 0,
        'blocked_sans_cc_p1':   c['blocked_sans_cc_p1'] > 0,
        'no_result_entreprise': c['no_result_entreprise'] > 0,
        'error_entreprise':     c['error_recherche_entreprise'] > 0,
        'select_localisation':  c['select_localisation'] > 0,
        'entreprise_sans_cc':   c['entreprise_sans_cc'] > 0,
        'select_cc_entreprise': c['select_cc_entreprise'] > 0,
        'particulier_employeur': c['select_particulier_employeur'] > 0,
        'modifier_entreprise':  c['click_modifier_entreprise'] > 0,
        'blocked_sans_cc_p2':   c['blocked_sans_cc_p2'] > 0,
        'modifier_cc':          c['click_modifier_cc'] > 0,
        'cc_non_traitee':       c['cc_non_traitee_retenue'] > 0,
        'lien_cc_externe':      c['click_lien_cc_externe'] > 0,
    })

steps_df = to_steps(counts_df)
steps_df[['idvisit', 'page', 'facade', 'device']] = counts_df[['idvisit', 'page', 'facade', 'device']]

def compute_funnel(steps):
    return steps.drop(columns=['idvisit', 'page', 'facade', 'device']).sum().astype(int)

funnel = compute_funnel(steps_df)
funnel

## 5. Graphique du funnel global

- Boîtes : nombre de parcours ayant atteint l'étape.
- Bleu : perte entre deux étapes consécutives (en % de l'étape précédente).
- Rouge : décrochages mesurés à cette étape (en % de l'étape) et retours arrière (flèche pointillée).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

nodes = {
    'Start':     ('Bloc affiché',                0.0,  0.0),
    'P1':        ('p1 · Je connais\nma CC',     2.0,  1.9),
    'P1_search': ('Recherche CC\nlancée',       4.0,  1.9),
    'P1_click':  ('Clic\n« Afficher »',         6.0,  1.9),
    'P1_ok':     ('Infos\naffichées',           8.0,  1.9),
    'P2':        ('p2 · Je cherche\nmon entreprise', 2.0, 0.0),
    'P2_search': ('Saisie\nentreprise',         4.0,  0.0),
    'P2_submit': ('Recherche\nsoumise',         6.0,  0.0),
    'P2_select': ('Entreprise\nchoisie',        8.0,  0.0),
    'P2_click':  ('Clic\n« Afficher »',        10.0,  0.0),
    'P2_ok':     ('Infos\naffichées',          12.0,  0.0),
    'P3':        ('p3 · Sans CC',                2.0, -1.9),
    'P3_click':  ('Clic\n« Afficher »',         4.0, -1.9),
    'P3_ok':     ('Infos\naffichées',           6.0, -1.9),
}
forward = [
    ('Start', 'P1'), ('P1', 'P1_search'), ('P1_search', 'P1_click'), ('P1_click', 'P1_ok'),
    ('Start', 'P2'), ('P2', 'P2_search'), ('P2_search', 'P2_submit'), ('P2_submit', 'P2_select'),
    ('P2_select', 'P2_click'), ('P2_click', 'P2_ok'),
    ('Start', 'P3'), ('P3', 'P3_click'), ('P3_click', 'P3_ok'),
]
start_edges = {('Start', 'P1'), ('Start', 'P2'), ('Start', 'P3')}

# Décrochages affichés sous une étape : (nœud, compteur, dénominateur, libellé)
drops = [
    ('Start',     'blocked_sans_option',  'Start',     'clic sans option'),
    ('P1_search', 'no_result_cc',         'P1_search', 'recherche sans résultat'),
    ('P1_click',  'blocked_sans_cc_p1',   'P1_click',  'bloqué : pas de CC'),
    ('P2_submit', 'no_result_entreprise', 'P2_submit', 'sans résultat'),
    ('P2_submit', 'error_entreprise',     'P2_submit', 'erreur API'),
    ('P2_select', 'entreprise_sans_cc',   'P2_select', 'entreprise sans CC'),
    ('P2_click',  'blocked_sans_cc_p2',   'P2_click',  'bloqué : pas de CC'),
]
# Retours arrière : (retour vers, depuis, compteur, dénominateur)
backward = [
    ('P2_submit', 'P2_select', 'modifier_entreprise', 'P2_select'),
]
# Apartés (gris) : (nœud, compteur, dénominateur, libellé)
asides = [
    ('Start', 'what_is_cc', 'Start', "« c'est quoi une CC ? »"),
]

BOX_W, BOX_H = 1.45, 0.8
FWD, BACK, ASIDE = '#2563eb', '#dc2626', '#64748b'
centers = {k: (x, y) for k, (_, x, y) in nodes.items()}

def anchors(src, dst):
    (x0, y0), (x1, y1) = centers[src], centers[dst]
    return (x0 + BOX_W/2, y0), (x1 - BOX_W/2, y1)

def at(p0, p1, t):
    return (p0[0] + t*(p1[0]-p0[0]), p0[1] + t*(p1[1]-p0[1]))

def pct(num, den):
    return num/den*100 if den else 0

def draw_funnel(ax, f, title):
    for key, (label, x, y) in nodes.items():
        ax.add_patch(FancyBboxPatch((x-BOX_W/2, y-BOX_H/2), BOX_W, BOX_H,
                     boxstyle="round,pad=0.02,rounding_size=0.08",
                     lw=1.2, edgecolor='#334155', facecolor='#f1f5f9', zorder=3))
        share = f"\n{pct(f[key], f['Start']):.1f}%" if key != 'Start' else ''
        ax.text(x, y, f"{label}\n{int(f[key])}{share}", ha='center', va='center',
                fontsize=8.5, fontweight='bold', zorder=4, linespacing=1.15)

    # flèches aller + % de perte
    for src, dst in forward:
        p0, p1 = anchors(src, dst)
        ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle='-|>', mutation_scale=14,
                     color=FWD, lw=1.6, zorder=2, shrinkA=0, shrinkB=0))
        if (src, dst) in start_edges:
            continue
        lx, ly = at(p0, p1, 0.5)
        ax.text(lx, ly, f"-{pct(f[src]-f[dst], f[src]):.1f}%", ha='center', va='center',
                color=FWD, fontsize=8, fontweight='bold',
                bbox=dict(boxstyle='round', fc='white', ec='none', alpha=0.9))

    # perte combinée au départ : parcours sans aucune radio cochée
    ax.text(0.0, -1.15, f"aucun choix\n-{pct(f['Start']-f['Any_route'], f['Start']):.1f}%",
            ha='center', va='center', color=FWD, fontsize=8, fontweight='bold',
            bbox=dict(boxstyle='round', fc='white', ec=FWD, alpha=0.95))

    # décrochages sous les étapes
    for key, ckey, dkey, label in drops:
        x, y = centers[key]
        offset = 0.62 + 0.28 * sum(1 for k, *_ in drops[:drops.index((key, ckey, dkey, label))] if k == key)
        ax.text(x, y - offset, f"× {label} : {pct(f[ckey], f[dkey]):.1f}%",
                ha='center', va='center', color=BACK, fontsize=7.5,
                bbox=dict(boxstyle='round', fc='white', ec='none', alpha=0.9))
    for key, ckey, dkey, label in asides:
        x, y = centers[key]
        ax.text(x, y + 0.62, f"{label} : {pct(f[ckey], f[dkey]):.1f}%",
                ha='center', va='center', color=ASIDE, fontsize=7.5,
                bbox=dict(boxstyle='round', fc='white', ec='none', alpha=0.9))

    # retours arrière
    for src, dst, bkey, dkey in backward:
        p0, p1 = anchors(src, dst)
        rad = 0.45
        ax.add_patch(FancyArrowPatch(p1, p0, arrowstyle='-|>', mutation_scale=12,
                     color=BACK, lw=1.3, ls='--', zorder=1, shrinkA=0, shrinkB=0,
                     connectionstyle=f"arc3,rad={rad}"))
        mx, my = (p0[0]+p1[0])/2, (p0[1]+p1[1])/2
        dx, dy = p0[0]-p1[0], p0[1]-p1[1]
        lx, ly = mx + 1.7*rad*dy, my - 1.7*rad*dx
        ax.text(lx, ly, f"↩ « Modifier » {pct(f[bkey], f[dkey]):.1f}%", ha='center', va='center',
                color=BACK, fontsize=7.5,
                bbox=dict(boxstyle='round', fc='white', ec='none', alpha=0.9))

    # bilan
    ax.text(12.0, -1.9,
            f"Infos affichées (toutes routes) : {int(f['OK_any'])} "
            f"soit {pct(f['OK_any'], f['Start']):.1f}% des blocs affichés\n"
            f"dont route non identifiée (pré-cochée) : {int(f['None_ok'])}\n"
            f"Clics « Afficher » : {int(f['Click_any'])} — "
            f"parcours bloqués au moins une fois : {int(f['blocked_sans_option'] + f['blocked_sans_cc_p1'] + f['blocked_sans_cc_p2'])}",
            ha='right', va='center', fontsize=8.5,
            bbox=dict(boxstyle='round', fc='#fefce8', ec='#ca8a04'))

    ax.set_xlim(-1, 13); ax.set_ylim(-3.1, 3.0); ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')

fig, ax = plt.subplots(figsize=(17, 7))
draw_funnel(ax, funnel, f"Funnel choix de la CC sur les contributions — {interval_start[:10]} → {interval_stop[:10]}")
plt.tight_layout()
plt.show()

## 6. Funnel par device (mobile vs desktop)

In [ ]:
funnels = pd.DataFrame({
    'mobile':  compute_funnel(steps_df[steps_df['device'] == 'mobile']),
    'desktop': compute_funnel(steps_df[steps_df['device'] == 'desktop']),
    'total':   funnel,
})

fig, axes = plt.subplots(2, 1, figsize=(17, 14))
draw_funnel(axes[0], funnels['mobile'],  "Funnel choix de la CC — MOBILE")
draw_funnel(axes[1], funnels['desktop'], "Funnel choix de la CC — DESKTOP")
plt.tight_layout()
plt.show()

## 7. Funnel par façade (fiche générique vs fiche CC)

Sur une fiche CC, le bloc n'est affiché qu'à l'ouverture explicite (arrivée externe ou
« Réinitialiser ») et le parcours peut être pré-coché : les deux funnels ne se comparent pas
directement.

In [ ]:
funnels_facade = pd.DataFrame({
    'générique': compute_funnel(steps_df[steps_df['facade'] == 'générique']),
    'fiche CC':  compute_funnel(steps_df[steps_df['facade'] == 'fiche CC']),
})

fig, axes = plt.subplots(2, 1, figsize=(17, 14))
draw_funnel(axes[0], funnels_facade['générique'], "Funnel choix de la CC — FICHE GÉNÉRIQUE")
draw_funnel(axes[1], funnels_facade['fiche CC'],  "Funnel choix de la CC — FICHE CC")
plt.tight_layout()
plt.show()

## 8. Taux de conversion par étape (vue « barres »)

Même funnel, lu en pourcentage des blocs affichés, une barre par étape et par parcours.

In [ ]:
branches = {
    'p1 · Je connais ma CC':        ['Start', 'P1', 'P1_search', 'P1_click', 'P1_ok'],
    'p2 · Je cherche mon entreprise': ['Start', 'P2', 'P2_search', 'P2_submit', 'P2_select', 'P2_click', 'P2_ok'],
    'p3 · Sans CC':                 ['Start', 'P3', 'P3_click', 'P3_ok'],
}
labels = {k: v[0].replace('\n', ' ') for k, v in nodes.items()}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=False)
for ax, (title, keys) in zip(axes, branches.items()):
    values = [funnel[k] for k in keys]
    rates  = [pct(v, funnel['Start']) for v in values]
    ax.barh(range(len(keys))[::-1], rates, color=FWD, alpha=0.85)
    ax.set_yticks(range(len(keys))[::-1], [labels[k] for k in keys], fontsize=8.5)
    for i, (v, r) in enumerate(zip(values, rates)):
        ax.text(r + 1, len(keys) - 1 - i, f"{v} ({r:.1f}%)", va='center', fontsize=8)
    ax.set_xlim(0, 115); ax.set_xlabel('% des blocs affichés'); ax.set_title(title, fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Décrochages et apartés

Chaque taux est rapporté au dénominateur pertinent (colonne `base`).

In [ ]:
drop_defs = [
    # (libellé, compteur, dénominateur)
    ("« C'est quoi une CC ? » cliqué",           'what_is_cc',           'Start'),
    ('Clic « Afficher » sans option cochée',     'blocked_sans_option',  'Start'),
    ('p1 · recherche CC sans résultat',          'no_result_cc',         'P1_search'),
    ('p1 · clic bloqué sans CC',                 'blocked_sans_cc_p1',   'P1_click'),
    ('p2 · recherche entreprise sans résultat',  'no_result_entreprise', 'P2_submit'),
    ('p2 · erreur API entreprises',              'error_entreprise',     'P2_submit'),
    ('p2 · localisation renseignée',             'select_localisation',  'P2_submit'),
    ('p2 · entreprise retenue sans CC',          'entreprise_sans_cc',   'P2_select'),
    ('p2 · choix parmi N conventions',           'select_cc_entreprise', 'P2_select'),
    ('p2 · carte « particulier employeur »',     'particulier_employeur', 'P2'),
    ('p2 · retour « Modifier » entreprise',      'modifier_entreprise',  'P2_select'),
    ('p2 · clic bloqué sans CC',                 'blocked_sans_cc_p2',   'P2_click'),
    ('p1+p2 · retour « Modifier » CC',           'modifier_cc',          'P1_click'),  # cf. note ci-dessous
    ('p1+p2 · CC retenue sans réponse',          'cc_non_traitee',       'Start'),
    ('→ clic vers la CC sur Légifrance',         'lien_cc_externe',      'cc_non_traitee'),
]

# « Modifier » CC est possible en p1 comme en p2 : on le rapporte aux parcours ayant retenu une CC
# (approximation : parcours p1 ou p2 avec au moins un clic « Afficher », ou une CC non traitée).
p1p2_with_cc = int(((steps_df['P1_click']) | (steps_df['P2_click']) | (steps_df['cc_non_traitee'])).sum())

rows = []
for label, ckey, dkey in drop_defs:
    den = p1p2_with_cc if ckey == 'modifier_cc' else funnel[dkey]
    base = 'P1|P2 avec CC' if ckey == 'modifier_cc' else dkey
    rows.append({'décrochage': label, 'parcours': int(funnel[ckey]), 'base': base,
                 'base_n': int(den), 'taux_%': round(pct(funnel[ckey], den), 1)})
drops_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(drops_df['décrochage'][::-1], drops_df['taux_%'][::-1], color=BACK, alpha=0.8)
for i, (t, n) in enumerate(zip(drops_df['taux_%'][::-1], drops_df['parcours'][::-1])):
    ax.text(t + 0.5, i, f"{t:.1f}% ({n})", va='center', fontsize=8)
ax.set_xlabel('% de la base'); ax.set_title('Décrochages et apartés', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

drops_df

## 10. Funnel par contribution

Une ligne par page (`name`), triée par nombre de blocs affichés. Les taux sont en % des blocs
affichés de la page. Rappel : sur une fiche CC ou une contribution sans réponse Code du travail,
le bouton « Afficher les informations » peut ne pas exister tant qu'aucune CC valide n'est retenue.

In [ ]:
by_page = steps_df.groupby('page').agg(
    facade=('facade', 'first'),
    blocs=('Start', 'sum'),
    p1=('P1', 'sum'), p2=('P2', 'sum'), p3=('P3', 'sum'),
    aucun_choix=('Any_route', lambda s: int((~s).sum())),
    clics=('Click_any', 'sum'),
    infos_affichees=('OK_any', 'sum'),
    bloques_sans_option=('blocked_sans_option', 'sum'),
    no_result_cc=('no_result_cc', 'sum'),
    no_result_entreprise=('no_result_entreprise', 'sum'),
    cc_non_traitee=('cc_non_traitee', 'sum'),
).sort_values('blocs', ascending=False)

for col in ['p1', 'p2', 'p3', 'aucun_choix', 'infos_affichees', 'bloques_sans_option', 'cc_non_traitee']:
    by_page[f'{col}_%'] = (by_page[col] / by_page['blocs'] * 100).round(1)

pd.set_option('display.max_colwidth', 90)
by_page.head(30)

## 11. Évolution quotidienne du taux de conversion

Part des parcours qui affichent les informations, par jour.

In [ ]:
first_ts = events_df.groupby(['idvisit', 'page'])['action_timestamp'].min()
steps_df['day'] = [first_ts[(v, p)].date() for v, p in zip(steps_df['idvisit'], steps_df['page'])]

daily = steps_df.groupby('day').agg(blocs=('Start', 'sum'), any_route=('Any_route', 'sum'),
                                    clics=('Click_any', 'sum'), ok=('OK_any', 'sum'))
daily['choix_%'] = (daily['any_route'] / daily['blocs'] * 100).round(1)
daily['clic_%']  = (daily['clics'] / daily['blocs'] * 100).round(1)
daily['ok_%']    = (daily['ok'] / daily['blocs'] * 100).round(1)

fig, ax = plt.subplots(figsize=(11, 4))
daily[['choix_%', 'clic_%', 'ok_%']].plot(ax=ax, marker='o')
ax.set_ylabel('% des blocs affichés'); ax.set_ylim(0); ax.grid(alpha=0.3)
ax.set_title('Conversion quotidienne : radio cochée / clic « Afficher » / infos affichées', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

daily

## 12. Bascules entre parcours (p1 ↔ p2 ↔ p3)

Au sein d'un même parcours (visite × page), l'usager peut cocher plusieurs radios successivement.
On reconstitue la **séquence des radios cochées** (doublons consécutifs ignorés) pour mesurer
combien explorent plusieurs options, et dans quel sens ils passent de l'une à l'autre.

In [ ]:
def route_sequence(actions):
    seq = []
    for a in actions:
        r = ROUTE_OF.get(a)
        if r and (not seq or seq[-1] != r):
            seq.append(r)
    return seq

routes = {}
for (idvisit, page), grp in events_df.groupby(['idvisit', 'page'], sort=False):
    routes[(idvisit, page)] = route_sequence(grp['action'].tolist())

routes_df = counts_df[['idvisit', 'page', 'facade', 'device']].copy()
routes_df['routes'] = [routes[(v, p)] for v, p in zip(routes_df['idvisit'], routes_df['page'])]
routes_df['n_radios']   = routes_df['routes'].str.len()
routes_df['n_bascules'] = (routes_df['n_radios'] - 1).clip(lower=0)
routes_df['explorees']  = routes_df['routes'].apply(lambda r: ' + '.join(sorted(set(r))) or 'aucune')
routes_df['premiere']   = routes_df['routes'].str[0]
routes_df['derniere']   = routes_df['routes'].str[-1]

with_choice = routes_df[routes_df['n_radios'] > 0]
multi = with_choice[with_choice['n_bascules'] > 0]
print(f"Parcours avec au moins une radio cochée : {len(with_choice)}")
print(f"  … dont au moins une bascule            : {len(multi)} ({pct(len(multi), len(with_choice)):.1f}%)")
print(f"  … dont ≥ 2 options distinctes explorées : "
      f"{int((with_choice['explorees'].str.contains('+', regex=False)).sum())} "
      f"({pct((with_choice['explorees'].str.contains('+', regex=False)).sum(), len(with_choice)):.1f}%)")
print()
print("Nombre de bascules par parcours :")
print(with_choice['n_bascules'].value_counts().sort_index().to_string())

In [ ]:
# Transitions consécutives (de → vers) sur l'ensemble des parcours
transitions = Counter()
for seq in with_choice['routes']:
    for a, b in zip(seq, seq[1:]):
        transitions[(a, b)] += 1
ROUTES = ['p1', 'p2', 'p3']
trans_df = pd.DataFrame([[transitions[(a, b)] for b in ROUTES] for a in ROUTES],
                        index=[f'depuis {r}' for r in ROUTES], columns=[f'vers {r}' for r in ROUTES])

# Première radio cochée → dernière radio cochée
first_last = pd.crosstab(with_choice['premiere'], with_choice['derniere'])

explored = with_choice['explorees'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

ax = axes[0]
im = ax.imshow(trans_df.values, cmap='Reds')
ax.set_xticks(range(3), trans_df.columns); ax.set_yticks(range(3), trans_df.index)
for i in range(3):
    for j in range(3):
        v = trans_df.values[i, j]
        ax.text(j, i, f"{v}", ha='center', va='center', fontsize=10,
                color='white' if v > trans_df.values.max()/2 else 'black')
ax.set_title('Bascules consécutives (nb)', fontsize=10, fontweight='bold')

ax = axes[1]
im = ax.imshow(first_last.values, cmap='Blues')
ax.set_xticks(range(len(first_last.columns)), [f'finit en {c}' for c in first_last.columns])
ax.set_yticks(range(len(first_last.index)), [f'commence en {i}' for i in first_last.index])
for i in range(len(first_last.index)):
    for j in range(len(first_last.columns)):
        v = first_last.values[i, j]
        ax.text(j, i, f"{v}\n{pct(v, first_last.values[i].sum()):.0f}%", ha='center', va='center', fontsize=9,
                color='white' if v > first_last.values.max()/2 else 'black')
ax.set_title('Première radio → dernière radio (parcours)', fontsize=10, fontweight='bold')

ax = axes[2]
ax.barh(explored.index[::-1], explored.values[::-1], color=FWD, alpha=0.85)
for i, v in enumerate(explored.values[::-1]):
    ax.text(v + explored.max()*0.01, i, f"{v} ({pct(v, len(with_choice)):.1f}%)", va='center', fontsize=8)
ax.set_xlim(0, explored.max()*1.3)
ax.set_title('Options explorées par parcours', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

trans_df

## 13. Que deviennent les usagers après une étape donnée ?

Pour chaque parcours ayant émis un event « jalon » (par exemple `no_result_cc`), on regarde **ce qui
se passe ensuite** : reste-t-il sur le même parcours, bascule-t-il vers une autre option, et
finit-il par afficher les informations ?

Cela permet de répondre à « les usagers qui ne trouvent pas leur CC en p1 vont-ils chercher leur
entreprise en p2 ? » plutôt que de les compter comme perdus.

In [ ]:
BLOCKED_OF = {'p1': 'blocked_sans_cc_p1', 'p2': 'blocked_sans_cc_p2', 'p3': None, 'none': 'blocked_sans_option'}

def success_in(c, route):
    blocked = BLOCKED_OF[route]
    return c[f'click_{route}'] > (c[blocked] if blocked else 0)

def outcome_after(actions, marker):
    """Devenir du parcours après la 1re occurrence de `marker` (None si absent)."""
    if marker not in actions:
        return None
    i = actions.index(marker)
    route = 'none'
    for a in actions[:i + 1]:
        route = ROUTE_OF.get(a, route)
    rest = actions[i + 1:]
    c = scan(rest, route)
    switch = next((r for r in route_sequence(rest) if r != route), None)
    ok = any(success_in(c, r) for r in ('p1', 'p2', 'p3', 'none'))
    where = f'bascule en {switch}' if switch else 'reste sur le parcours'
    return f"{where} · {'infos affichées' if ok else 'abandon'}"

MARKERS = {
    'start_recherche_cc':   'p1 · recherche CC lancée',
    'no_result_cc':         'p1 · recherche CC sans résultat',
    'blocked_sans_cc_p1':   'p1 · clic bloqué sans CC',
    'submit_recherche_entreprise': 'p2 · recherche entreprise soumise',
    'no_result_entreprise': 'p2 · recherche entreprise sans résultat',
    'entreprise_sans_cc':   'p2 · entreprise retenue sans CC',
    'blocked_sans_cc_p2':   'p2 · clic bloqué sans CC',
    'blocked_sans_option':  'clic « Afficher » sans option cochée',
}

actions_by_parcours = {k: grp['action'].tolist()
                       for k, grp in events_df.groupby(['idvisit', 'page'], sort=False)}
kept_keys = set(zip(counts_df['idvisit'], counts_df['page']))

rows = []
for marker, label in MARKERS.items():
    for key, actions in actions_by_parcours.items():
        if key not in kept_keys:
            continue
        o = outcome_after(actions, marker)
        if o:
            rows.append({'jalon': label, 'devenir': o})
outcomes_df = pd.DataFrame(rows)

table = pd.crosstab(outcomes_df['jalon'], outcomes_df['devenir'])
table = table.reindex(MARKERS.values())
table_pct = (table.div(table.sum(axis=1), axis=0) * 100).round(1)

# Couleurs fixes : vert = infos affichées, rouge = abandon ; teinte claire = reste sur le parcours,
# teintes foncées = bascule vers une autre option (ce que l'on cherche à voir)
PALETTE = {
    'reste sur le parcours · infos affichées': '#bbf7d0',
    'bascule en p1 · infos affichées':         '#15803d',
    'bascule en p2 · infos affichées':         '#22c55e',
    'bascule en p3 · infos affichées':         '#4ade80',
    'reste sur le parcours · abandon':         '#fecaca',
    'bascule en p1 · abandon':                 '#991b1b',
    'bascule en p2 · abandon':                 '#dc2626',
    'bascule en p3 · abandon':                 '#f87171',
}
WHITE_TEXT = {'#15803d', '#22c55e', '#991b1b', '#dc2626'}
cols = [c for c in PALETTE if c in table.columns]

fig, ax = plt.subplots(figsize=(13, 5.5))
table_pct[cols].iloc[::-1].plot.barh(stacked=True, ax=ax, color=[PALETTE[c] for c in cols], width=0.7)
for i, (_, row) in enumerate(table_pct[cols].iloc[::-1].iterrows()):
    left = 0
    for c in cols:
        v = row[c]
        if v >= 3:
            ax.text(left + v/2, i, f"{v:.0f}%", ha='center', va='center', fontsize=7.5,
                    color='white' if PALETTE[c] in WHITE_TEXT else 'black')
        left += v
ax.set_xlim(0, 100); ax.set_xlabel('% des parcours ayant émis le jalon'); ax.set_ylabel('')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=8, frameon=False)
ax.set_title("Devenir des parcours après un jalon — vert : infos affichées, rouge : abandon",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

table.assign(total=table.sum(axis=1))

## 14. Zoom sur les parcours sans aucun choix

Trois quarts des blocs affichés ne donnent lieu à aucune radio cochée. Pour comprendre qui sont ces
usagers, on récupère les **pages vues** des visites concernées et on regarde :

- la part de parcours où la **visite ne comprend que cette page** (aucune autre page vue) ;
- pour ces visites mono-page, le **temps passé** (durée de la visite, qui inclut les events) ;
- la **provenance** : page interne du site (la contribution n'est pas la page d'entrée), accès
  direct, moteur de recherche, assistant IA, site externe, réseau social.

Les parcours **avec** choix sont affichés en regard, comme point de comparaison.

In [ ]:
from urllib.parse import urlparse

query_pages = f"""
    SELECT idvisit, action_id, action_url, action_timespent, visitduration, referrertype, referrername
    FROM matomo_partitioned
    WHERE action_timestamp >= '{interval_start}'
      AND action_timestamp <  '{interval_stop}'
      AND action_type = 'action'
      AND idvisit IN (
          SELECT DISTINCT idvisit FROM matomo_partitioned
          WHERE action_timestamp >= '{interval_start}'
            AND action_timestamp <  '{interval_stop}'
            AND action_eventcategory = 'cc_search_funnel'
      )
"""
with MatomoSQLConnectorSync() as matomo:
    pages_df = matomo.run_query_df(query_pages)

pages_df['seq']  = pages_df['action_id'].str.rsplit('_', n=1).str[1].astype(int)
pages_df['path'] = pages_df['action_url'].str.split('?').str[0].apply(lambda u: urlparse(u).path.strip('/'))
pages_df = pages_df.sort_values(['idvisit', 'seq'])

visits = pages_df.groupby('idvisit').agg(
    n_pages=('path', 'size'),
    n_paths=('path', 'nunique'),
    first_path=('path', 'first'),
    visitduration=('visitduration', lambda s: pd.to_numeric(s, errors='coerce').max()),
    referrertype=('referrertype', 'first'),
    referrername=('referrername', 'first'),
)
print(f"{len(pages_df)} pages vues sur {len(visits)} visites")

In [ ]:
REFERRER_LABEL = {'direct': 'accès direct', 'search': 'moteur de recherche', 'ai': 'assistant IA',
                  'website': 'site externe', 'social': 'réseau social', 'campaign': 'campagne'}

ctx = steps_df[['idvisit', 'page', 'Any_route']].merge(visits, left_on='idvisit', right_index=True, how='left')
ctx['groupe']      = ctx['Any_route'].map({False: 'aucun choix', True: 'avec choix'})
ctx['mono_page']   = (ctx['n_paths'] == 1) & (ctx['first_path'] == ctx['page'])
ctx['entree']      = ctx['first_path'] == ctx['page']
ctx['provenance']  = ctx['referrertype'].map(REFERRER_LABEL).fillna('inconnue')
ctx.loc[~ctx['entree'], 'provenance'] = 'page interne du site'
missing = ctx['n_pages'].isna().sum()
if missing:
    print(f"{missing} parcours sans page vue retrouvée (visite hors intervalle ?) — ignorés dans les taux")
ctx = ctx.dropna(subset=['n_pages'])

no_choice = ctx[ctx['groupe'] == 'aucun choix']
print(f"Parcours sans aucun choix : {len(no_choice)}")
print(f"  … visite mono-page        : {int(no_choice['mono_page'].sum())} ({pct(no_choice['mono_page'].sum(), len(no_choice)):.1f}%)")
print(f"  … page d'entrée de la visite : {int(no_choice['entree'].sum())} ({pct(no_choice['entree'].sum(), len(no_choice)):.1f}%)")

mono = ctx[ctx['mono_page']]
print()
print("Durée de visite (s) des visites mono-page :")
print(mono.groupby('groupe')['visitduration'].describe(percentiles=[.25, .5, .75]).round(0).to_string())

In [ ]:
DURATION_BINS   = [-1, 10, 30, 60, 180, 600, float('inf')]
DURATION_LABELS = ['< 10 s', '10–30 s', '30–60 s', '1–3 min', '3–10 min', '> 10 min']
mono = mono.assign(duree=pd.cut(mono['visitduration'], bins=DURATION_BINS, labels=DURATION_LABELS))

mono_rate = ctx.groupby('groupe')['mono_page'].mean() * 100
duree_pct = pd.crosstab(mono['duree'], mono['groupe'], normalize='columns') * 100
prov_pct  = pd.crosstab(ctx['provenance'], ctx['groupe'], normalize='columns') * 100
prov_pct  = prov_pct.loc[prov_pct['aucun choix'].sort_values(ascending=False).index]
GROUP_COLORS = {'aucun choix': BACK, 'avec choix': FWD}

fig, axes = plt.subplots(1, 3, figsize=(17, 5), gridspec_kw={'width_ratios': [1, 1.4, 1.6]})

ax = axes[0]
bars = ax.bar(mono_rate.index, mono_rate.values, color=[GROUP_COLORS[g] for g in mono_rate.index], alpha=0.85, width=0.6)
for b, v, g in zip(bars, mono_rate.values, mono_rate.index):
    ax.text(b.get_x() + b.get_width()/2, v + 1, f"{v:.1f}%\n({int(ctx[ctx['groupe'] == g]['mono_page'].sum())})",
            ha='center', va='bottom', fontsize=9)
ax.set_ylim(0, 100); ax.set_ylabel('% des parcours')
ax.set_title('Visite limitée à cette page', fontsize=10, fontweight='bold')

ax = axes[1]
duree_pct.plot.bar(ax=ax, color=[GROUP_COLORS[g] for g in duree_pct.columns], alpha=0.85, width=0.75)
ax.set_ylabel('% des visites mono-page'); ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0, labelsize=8); ax.legend(fontsize=8, frameon=False)
ax.set_title('Temps passé (visites mono-page)', fontsize=10, fontweight='bold')

ax = axes[2]
prov_pct.iloc[::-1].plot.barh(ax=ax, color=[GROUP_COLORS[g] for g in prov_pct.columns], alpha=0.85, width=0.75)
for i, (_, row) in enumerate(prov_pct.iloc[::-1].iterrows()):
    ax.text(row.max() + 1, i, f"{row['aucun choix']:.1f}% / {row['avec choix']:.1f}%", va='center', fontsize=7.5)
ax.set_xlim(0, prov_pct.values.max() * 1.35); ax.set_xlabel('% des parcours'); ax.set_ylabel('')
ax.legend(fontsize=8, frameon=False, loc='lower right')
ax.set_title("Provenance (page interne vs entrée directe sur la page)", fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

pd.DataFrame({'aucun choix (n)': pd.crosstab(ctx['provenance'], ctx['groupe'])['aucun choix'],
              'aucun choix (%)': prov_pct['aucun choix'].round(1),
              'avec choix (n)':  pd.crosstab(ctx['provenance'], ctx['groupe'])['avec choix'],
              'avec choix (%)':  prov_pct['avec choix'].round(1)}).loc[prov_pct.index]

Pour les entrées depuis un **site externe**, les sites qui envoient le plus d'usagers n'ayant fait aucun choix :

In [ ]:
ext = no_choice[no_choice['provenance'] == 'site externe']
ext['referrername'].value_counts().head(15).rename('parcours sans choix').to_frame()